# Validate the Reference H2O Bundle

Validate required files, manifest fields, feature order, immutable version, H2O version, checksums, and golden prediction parity before any Azure registration.

**Source:** Adapted from this repository's `notebooks/h2o_mojo/02_onboard_customer_mojo.ipynb`.

In [1]:
from pathlib import Path
import json
import os
import sys

import h2o
import numpy as np
import pandas as pd
from dotenv import load_dotenv

notebook_file = globals().get("__vsc_ipynb_file__")
search_start = (
    Path(notebook_file).resolve().parent
    if notebook_file
    else Path.cwd().resolve()
)
for candidate in (search_start, *search_start.parents):
    if (candidate / ".env.example").is_file() and (candidate / "pipelines").is_dir():
        WORKSHOP_ROOT = candidate
        break
else:
    raise FileNotFoundError("Run this notebook from inside the workshop folder")
load_dotenv(WORKSHOP_ROOT / ".env", override=True)

bundle_value = Path(os.environ["H2O_BUNDLE_DIR"])
BUNDLE_DIR = bundle_value if bundle_value.is_absolute() else WORKSHOP_ROOT / bundle_value
sys.path.insert(0, str(WORKSHOP_ROOT / "src/h2o"))
from validate_bundle import validate_bundle

summary = validate_bundle(BUNDLE_DIR, os.environ["H2O_VERSION"])
display(summary)
manifest = json.loads((BUNDLE_DIR / "model_manifest.json").read_text(encoding="utf-8"))
golden_input = pd.read_csv(BUNDLE_DIR / "golden_input.csv")
golden_expected = pd.read_csv(BUNDLE_DIR / "golden_expected.csv")

try:
    h2o.init(max_mem_size="2G", nthreads=-1)
    model = h2o.load_model(str(BUNDLE_DIR / manifest["model_file"]))
    frame = h2o.H2OFrame(golden_input)
    for column in manifest["categorical_features"]:
        frame[column] = frame[column].asfactor()
    actual = model.predict(frame).as_data_frame()
    np.testing.assert_allclose(golden_expected["predict"], actual["predict"], rtol=1e-6, atol=1e-6)
    comparison = pd.DataFrame({"expected": golden_expected["predict"], "actual": actual["predict"]})
    display(comparison.head(10))
    print("Bundle validation and golden prediction parity passed.")
finally:
    if h2o.connection() is not None:
        h2o.cluster().shutdown(prompt=False)

{'bundle_dir': '/home/azureuser/MLOPs-AzureML-workshop-277d1b6/workshop/data/h2o/reference_bundle',
 'model_name': 'taxi-fare-h2o-binary',
 'model_version': '1',
 'model_format': 'h2o_binary',
 'h2o_version': '3.46.0.12',
 'runtime_h2o_version': '3.46.0.12',
 'mojo_version': None,
 'model_category': None,
 'model_file': 'taxi-fare-gbm',
 'features': ['vendorID',
  'passengerCount',
  'tripDistance',
  'paymentType',
  'pickupHour'],
 'golden_provided': True,
 'golden_validation_enabled': True,
 'golden_required': False,
 'golden_input_file': 'golden_input.csv',
 'golden_expected_file': 'golden_expected.csv',
 'golden_rows': 20,
 'checksums': 'passed',
 'packaging_validation': 'not_recorded',
 'golden_validation': 'not_recorded'}

Checking whether there is an H2O instance running at http://localhost:54321..... not found.
Attempting to start a local H2O server...
  Java Version: openjdk version "17.0.18-internal" 2026-01-20; OpenJDK Runtime Environment (build 17.0.18-internal+0-adhoc.rattler.src); OpenJDK 64-Bit Server VM (build 17.0.18-internal+0-adhoc.rattler.src, mixed mode, sharing)
  Starting server from /anaconda/envs/azureml-workshop/lib/python3.12/site-packages/h2o/backend/bin/h2o.jar
  Ice root: /tmp/tmpksh3dpf0
  JVM stdout: /tmp/tmpksh3dpf0/h2o_azureuser_started_from_python.out
  JVM stderr: /tmp/tmpksh3dpf0/h2o_azureuser_started_from_python.err
  Server is running at http://127.0.0.1:54321
Connecting to H2O server at http://127.0.0.1:54321 ... successful.


H2O_cluster_uptime:,01 secs
H2O_cluster_timezone:,Etc/UTC
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.12
H2O_cluster_version_age:,1 month and 4 days
H2O_cluster_name:,H2O_from_python_azureuser_cljc3f
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,2 Gb
H2O_cluster_total_cores:,8
H2O_cluster_allowed_cores:,8
H2O_cluster_status:,"locked, healthy"



+------------------------------------------------------------------+
| You are running the community edition of H2O-3 OSS.              |
|                                                                  |
| For commercial use, H2O-3 Secure is now recommended.             |
| This includes production support, CVE fixes, multi-node scaling, |
| model artifact extraction, and more.                             |
| See h2o.ai/h2o-3/oss-vs-secure for additional details.           |
| Contact enterprise@h2o.ai to upgrade.                            |
+------------------------------------------------------------------+
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
gbm prediction progress: |███████████████████████████████████████████████████████| (done) 100%


/anaconda/envs/azureml-workshop/lib/python3.12/site-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


,expected,actual
0,34.823667,34.823667
1,20.211054,20.211054
2,13.282541,13.282541
3,9.505314,9.505314
4,7.076866,7.076866
5,18.151701,18.151701
6,11.130871,11.130871
7,6.696699,6.696699
8,7.701867,7.701867
9,12.329862,12.329862


Bundle validation and golden prediction parity passed.
H2O session _sid_b963 closed.


## Expected Result

The reference bundle passes schema, immutable-version, checksum, exact-H2O-version, feature-order, and golden-parity validation.

Next: `03_test_local_endpoint.ipynb`.